# SentimentFlow — Notebook 01: Análisis de Sentimiento

Este notebook es invocado por **Azure Data Factory** cada vez que el consumer deposita un nuevo lote de reseñas (`.jsonl`) en el contenedor `raw-reviews` de Azure Blob Storage.

**Flujo:**
1. Lee el fichero `.jsonl` desde el parámetro `input_file`.
2. Limpia y normaliza el texto de cada reseña.
3. Aplica análisis de sentimiento con **VADER** (Valence Aware Dictionary and sEntiment Reasoner).
4. Calcula el sentimiento compuesto (`compound`) y clasifica en Positivo / Neutro / Negativo.
5. Escribe el resultado en formato Parquet en el contenedor `processed-reviews`.

**Responsable:** Persona 2

In [ ]:
# ─── Parámetros inyectados por ADF ────────────────────────────────────────────
# En Databricks, ADF pasa los parámetros como widgets.
# Valores por defecto para ejecución manual/test.
dbutils.widgets.text('raw_container',       'raw-reviews')
dbutils.widgets.text('processed_container', 'processed-reviews')
dbutils.widgets.text('input_file',          'batch_20260430T120000_0100.jsonl')
dbutils.widgets.text('run_date',            '2026-04-30')

RAW_CONTAINER       = dbutils.widgets.get('raw_container')
PROCESSED_CONTAINER = dbutils.widgets.get('processed_container')
INPUT_FILE          = dbutils.widgets.get('input_file')
RUN_DATE            = dbutils.widgets.get('run_date')

print(f'Fichero de entrada : {RAW_CONTAINER}/{INPUT_FILE}')
print(f'Contenedor salida  : {PROCESSED_CONTAINER}')
print(f'Fecha de ejecución : {RUN_DATE}')

In [ ]:
# ─── Librerías ────────────────────────────────────────────────────────────────
# En Databricks Runtime 13+, PySpark y pandas están preinstalados.
# VADER se instala en el clúster mediante la UI de Libraries o con %pip.
%pip install vaderSentiment --quiet

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType, DoubleType, TimestampType
)
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
import re

spark = SparkSession.builder.getOrCreate()
print('Spark version:', spark.version)

In [ ]:
# ─── Montar Azure Blob Storage via ABFS ───────────────────────────────────────
# Requiere que las credenciales del Storage Account estén configuradas
# en Databricks Secrets o en las propiedades del clúster.
STORAGE_ACCOUNT = dbutils.secrets.get(scope='sentimentflow', key='storage_account_name')
STORAGE_KEY      = dbutils.secrets.get(scope='sentimentflow', key='storage_account_key')

spark.conf.set(
    f'fs.azure.account.key.{STORAGE_ACCOUNT}.blob.core.windows.net',
    STORAGE_KEY
)

RAW_PATH       = f'wasbs://{RAW_CONTAINER}@{STORAGE_ACCOUNT}.blob.core.windows.net/{INPUT_FILE}'
PROCESSED_PATH = f'wasbs://{PROCESSED_CONTAINER}@{STORAGE_ACCOUNT}.blob.core.windows.net/{RUN_DATE}/'

print('Raw path      :', RAW_PATH)
print('Processed path:', PROCESSED_PATH)

In [ ]:
# ─── 1. Lectura del lote raw ───────────────────────────────────────────────────
schema = StructType([
    StructField('review_id',    StringType(),    False),
    StructField('product_id',   StringType(),    False),
    StructField('product_name', StringType(),    True),
    StructField('category',     StringType(),    True),
    StructField('user_id',      StringType(),    True),
    StructField('rating',       IntegerType(),   True),
    StructField('review_text',  StringType(),    True),
    StructField('country',      StringType(),    True),
    StructField('timestamp',    StringType(),    True),
])

df_raw = spark.read.schema(schema).json(RAW_PATH)
print(f'Registros leídos: {df_raw.count()}')
df_raw.show(5, truncate=60)

In [ ]:
# ─── 2. Limpieza y normalización ──────────────────────────────────────────────
def clean_text(text: str) -> str:
    if not text:
        return ''
    text = text.lower().strip()
    text = re.sub(r'[^\w\s.,!?áéíóúüñ]', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text

clean_text_udf = F.udf(clean_text, StringType())

df_clean = (
    df_raw
    .withColumn('review_text_clean', clean_text_udf(F.col('review_text')))
    .withColumn('event_ts', F.to_timestamp(F.col('timestamp')))
    .filter(F.col('review_id').isNotNull())
    .filter(F.col('rating').between(1, 5))
    .dropDuplicates(['review_id'])
)

print(f'Registros tras limpieza: {df_clean.count()}')

In [ ]:
# ─── 3. Análisis de sentimiento con VADER ─────────────────────────────────────
# VADER devuelve scores: neg, neu, pos y compound (-1.0 a 1.0)
# Thresholds estándar: compound >= 0.05 → Positivo
#                      compound <= -0.05 → Negativo
#                      en otro caso     → Neutro

analyzer = SentimentIntensityAnalyzer()

def get_compound(text: str) -> float:
    if not text:
        return 0.0
    return float(analyzer.polarity_scores(text)['compound'])

def get_label(compound: float) -> str:
    if compound is None:
        return 'Neutro'
    if compound >= 0.05:
        return 'Positivo'
    if compound <= -0.05:
        return 'Negativo'
    return 'Neutro'

compound_udf = F.udf(get_compound, DoubleType())
label_udf    = F.udf(get_label,    StringType())

df_sentiment = (
    df_clean
    .withColumn('sentiment_score', compound_udf(F.col('review_text_clean')))
    .withColumn('sentiment_label', label_udf(F.col('sentiment_score')))
    .withColumn('processed_at', F.current_timestamp())
)

df_sentiment.select(
    'review_id', 'product_name', 'rating',
    'sentiment_score', 'sentiment_label'
).show(10, truncate=40)

In [ ]:
# ─── 4. Estadísticas del lote ─────────────────────────────────────────────────
print('Distribución de sentimiento:')
df_sentiment.groupBy('sentiment_label').count().orderBy('count', ascending=False).show()

print('\nSentimiento medio por categoría:')
df_sentiment.groupBy('category').agg(
    F.round(F.avg('sentiment_score'), 4).alias('avg_sentiment'),
    F.round(F.avg('rating'), 2).alias('avg_rating'),
    F.count('*').alias('total_reviews')
).orderBy('avg_sentiment', ascending=False).show()

In [ ]:
# ─── 5. Escritura en Parquet (zona processed) ─────────────────────────────────
OUTPUT_COLS = [
    'review_id', 'product_id', 'product_name', 'category',
    'user_id', 'rating', 'review_text_clean', 'country',
    'event_ts', 'sentiment_score', 'sentiment_label', 'processed_at'
]

df_output = df_sentiment.select(*OUTPUT_COLS)

(
    df_output
    .repartition(1)          # Un solo fichero Parquet por lote (ADF lo espera así)
    .write
    .mode('append')
    .option('compression', 'snappy')
    .parquet(PROCESSED_PATH)
)

print(f'✓ Parquet escrito en: {PROCESSED_PATH}')
print(f'  Registros procesados: {df_output.count()}')

# Retorna metadatos al pipeline de ADF
dbutils.notebook.exit(f'OK|{df_output.count()}|{PROCESSED_PATH}')